<a href="https://colab.research.google.com/github/farankhandev/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farankhandev/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1 : AI sessions and SEO performance

The research paper reports findings about direct click-through visits from AI tools to tracked blog pages. The paper defines AI sessions as direct click-through visits from AI tools and distinguishes these from impressions or citations inside AI systems.

**My methodology question:**
How were AI sessions identified and attributed to AI tools, and how was the outcome defined? I would want to understand whether the measurement captures all relevant AI-driven traffic or only the traffic that can be directly observed and attributed.

### Finding 2 : SEO findings based on recent performance data

The paper reports several SEO findings using the most recent 90 complete days of data, while some structural analyses use the full available warehouse history.

**My methodology question:**
Does the validation and time window support the strength of each claim? In particular, I would want to know whether the observed relationships were tested across different time periods or whether they could be specific to the selected 90-day window.

### Why I am asking these questions

These are not claims that the research is incorrect. They are questions I would ask when reviewing the methodology of my own work as well. Understanding how the labels or outcomes are defined and whether the validation design supports the conclusion is important before treating an observed relationship as evidence of a broader effect.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
!git clone https://github.com/farankhandev/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 183 (delta 84), reused 103 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (183/183), 1.87 MiB | 10.23 MiB/s, done.
Resolving deltas: 100% (84/84), done.


In [ ]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship/flyrank-ml-internship


In [ ]:
import os

print(os.path.exists("data/raw/content_refresh_anonymized.csv"))

True


In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df_model = df[df["impressions_90d"] >= 10].copy()

print("Original rows:", len(df))
print("Rows after filtering:", len(df_model))
print("Unique clients:", df_model["client_id"].nunique())

Original rows: 30000
Rows after filtering: 26254
Unique clients: 31


In [ ]:
df_section3 = df_model.copy()

ctr_score = 1 - (
    (df_section3["ctr"] - df_section3["ctr"].min()) /
    (df_section3["ctr"].max() - df_section3["ctr"].min())
)

position_score = (
    (df_section3["avg_position"] - df_section3["avg_position"].min()) /
    (df_section3["avg_position"].max() - df_section3["avg_position"].min())
)

df_section3["baseline_score"] = (
    ctr_score * 50 +
    position_score * 50
)

print("Week-4 baseline score recreated.")
print(df_section3[[
    "content_id",
    "ctr",
    "avg_position",
    "baseline_score"
]].head())

Week-4 baseline score recreated.
             content_id   ctr  avg_position  baseline_score
0  content_304f48230142  0.76          10.6       54.140284
1  content_a1fb4e703a9e  0.05          20.3       60.212870
2  content_9aa793d4d895  0.09          36.5       68.362881
3  content_331d6c4de07b  0.49           6.2       52.347786
4  content_d99b7a2d90ca  0.13          44.0       72.101129


In [ ]:
df_model = df_section3.copy()

In [ ]:
print(df_model.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'baseline_score']


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

features = [
    "impressions_90d",
    "engagement_rate",
    "search_volume",
    "cpc",
    "competition",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "trend_pct"
]

X = df_model[features]
y = df_model["baseline_score"]
groups = df_model["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print(
    "Training clients:",
    df_model.iloc[train_idx]["client_id"].nunique()
)

print(
    "Testing clients:",
    df_model.iloc[test_idx]["client_id"].nunique()
)

Training rows: 21348
Testing rows: 4906
Training clients: 24
Testing clients: 7


In [ ]:
from sklearn.tree import DecisionTreeRegressor

model_honest = DecisionTreeRegressor(
    random_state=42
)

model_honest.fit(X_train, y_train)

print("Decision Tree trained successfully on the honest client-grouped split.")

Decision Tree trained successfully on the honest client-grouped split.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred_honest = model_honest.predict(X_test)

honest_mae = mean_absolute_error(y_test, y_pred_honest)
honest_rmse = np.sqrt(mean_squared_error(y_test, y_pred_honest))
honest_r2 = r2_score(y_test, y_pred_honest)

print("Honest client-grouped split results:")
print(f"MAE:  {honest_mae:.4f}")
print(f"RMSE: {honest_rmse:.4f}")
print(f"R²:   {honest_r2:.4f}")

Honest client-grouped split results:
MAE:  7.1270
RMSE: 10.5331
R²:   -1.8750


In [ ]:
comparison = pd.DataFrame({
    "Validation": [
        "Week-5 random split",
        "ML-09 client-grouped split"
    ],
    "MAE": [
        5.0102,
        honest_mae
    ],
    "RMSE": [
        6.9601,
        honest_rmse
    ],
    "R2": [
        0.2101,
        honest_r2
    ]
})

comparison

,Validation,MAE,RMSE,R2
0,Week-5 random split,5.010200,6.960100,0.210100
1,ML-09 client-grouped split,7.127003,10.533127,-1.874984


### Interpretation

The client-grouped split produced weaker measured performance than the Week-5 random split. MAE increased from 5.01 to 7.13, while RMSE increased from 6.96 to 10.53. The R² score also decreased from 0.21 to -1.87.

This suggests that the model's performance was less reliable when tested on content from clients that were not represented in the training data. The random split may have produced a more favorable estimate because content from the same clients could appear in both the training and test sets.

The client-grouped result is a more conservative measure of generalization to unseen clients. These results are directional and should be treated as decision-support rather than evidence that the model will perform the same way on all future clients.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
target = "baseline_score"

leakage_checks = []

for feature in features:
    leakage_checks.append({
        "feature": feature,
        "is_target": feature == target,
        "contains_target_name": target.lower() in feature.lower(),
        "is_future_outcome": feature in [
            "clicks_90d",
            "pageviews_90d",
            "sessions_90d",
            "users_90d",
            "engaged_sessions_90d",
            "ai_sessions_90d",
            "scroll_events_90d"
        ]
    })

leakage_audit = pd.DataFrame(leakage_checks)

leakage_audit

,feature,is_target,contains_target_name,is_future_outcome
0,impressions_90d,False,False,False
1,engagement_rate,False,False,False
2,search_volume,False,False,False
3,cpc,False,False,False
4,competition,False,False,False
5,word_count,False,False,False
6,char_count,False,False,False
7,content_age_days,False,False,False
8,days_since_last_update,False,False,False
9,trend_pct,False,False,False


In [ ]:
leakage_corr = (
    df_model[features + ["baseline_score"]]
    .corr(numeric_only=True)["baseline_score"]
    .drop("baseline_score")
    .sort_values(key=abs, ascending=False)
)

print("Feature correlations with baseline_score:")
print(leakage_corr)

Feature correlations with baseline_score:
content_age_days          0.159120
word_count                0.127730
char_count                0.104171
impressions_90d          -0.095547
competition               0.061334
cpc                       0.058024
engagement_rate          -0.051270
search_volume             0.047990
trend_pct                 0.043751
days_since_last_update    0.038443
Name: baseline_score, dtype: float64


In [ ]:
feature_timing = pd.DataFrame({
    "feature": features,
    "available_before_prediction": [
        True,  # impressions_90d
        True,  # engagement_rate
        True,  # search_volume
        True,  # cpc
        True,  # competition
        True,  # word_count
        True,  # char_count
        True,  # content_age_days
        True,  # days_since_last_update
        True   # trend_pct
    ]
})

feature_timing

,feature,available_before_prediction
0,impressions_90d,True
1,engagement_rate,True
2,search_volume,True
3,cpc,True
4,competition,True
5,word_count,True
6,char_count,True
7,content_age_days,True
8,days_since_last_update,True
9,trend_pct,True


### Leakage audit conclusion

The final feature set does not directly contain the target `baseline_score`, and none of the selected features were flagged as future-outcome variables in the initial audit. The feature-to-target correlations were also modest, with the largest observed correlation being `content_age_days` at approximately 0.16.

Based on these checks, I did not observe an obvious leakage issue in the final feature names or correlations. However, correlation alone cannot prove that a feature is leakage-free. The feature timing should also be verified against the actual data-generation process before treating the model as fully leakage-safe.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

### Original claim

The model can accurately predict content performance and provide reliable predictions for future content.

### Revised claim

The model performed reasonably on the random split, but its performance dropped when I tested it on completely unseen clients. This suggests that the random split may have given a more optimistic view of the model's performance. Based on these results, I would treat the model as a directional decision-support tool rather than claiming that it can reliably predict performance for new clients.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] The Week-5 random split is compared with the ML-09 client-grouped split
- [x] The leakage audit checks the final feature set
- [x] The notebook includes a real validation result and interpretation
- [x] Committed to my repo under `work/notebooks/`